In [54]:
import os
import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

# Veri setini yükle
CSV_FILENAME = "../datasets/bankloan.csv"
df = pd.read_csv(CSV_FILENAME)
print(f"Veri başarıyla yüklendi. Boyut: {df.shape}")

Veri başarıyla yüklendi. Boyut: (5000, 14)


In [41]:
# ==========================================
# 1.1 İLK KEŞİF
# ==========================================

print("\n" + "=" * 50)
print("🔍 EKSİK VERİ ANALİZİ (Missing Values)")
print("=" * 50)

# Sadece eksik verisi olan sütunları ve oranlarını hesapla
missing_count = df.isnull().sum()
missing_percent = 100 * df.isnull().mean()

missing_df = pd.DataFrame(
    {"Eksik Sayısı": missing_count, "Oran (%)": missing_percent}
)
# Sadece eksik değeri olanları filtrele ve büyükten küçüğe sırala
missing_df = missing_df[missing_df["Eksik Sayısı"] > 0].sort_values(
    by="Eksik Sayısı", ascending=False
)

if missing_df.empty:
  print("✨ Harika! Veri setinde hiç eksik değer yok.")
else:
  print(missing_df.to_string())

print("=" * 50 + "\n")


🔍 EKSİK VERİ ANALİZİ (Missing Values)
✨ Harika! Veri setinde hiç eksik değer yok.



In [42]:
# ==========================================
# 1.2 KEŞİFSEL VERİ ANALİZİ
# ==========================================
# 1. Genel Yapı ve Tipler
print("--- Bilgiler ve Tipler ---")
print(df.info())

# 2. İstatistiksel Dağılım
print("\n--- İstatistiksel Özet ---")
display(df.describe())

# 3. Eksik Değer Kontrolü
print("\n--- Eksik Değerler ---")
missing = df.isnull().sum()
print(missing[missing > 0])

# 4. İlk 5 Satır Göz Atma
print("\n--- İlk 5 Satır ---")
display(df.head())

--- Bilgiler ve Tipler ---
<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 14 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   ID                  5000 non-null   int64  
 1   Age                 5000 non-null   int64  
 2   Experience          5000 non-null   int64  
 3   Income              5000 non-null   int64  
 4   ZIP.Code            5000 non-null   int64  
 5   Family              5000 non-null   int64  
 6   CCAvg               5000 non-null   float64
 7   Education           5000 non-null   int64  
 8   Mortgage            5000 non-null   int64  
 9   Personal.Loan       5000 non-null   int64  
 10  Securities.Account  5000 non-null   int64  
 11  CD.Account          5000 non-null   int64  
 12  Online              5000 non-null   int64  
 13  CreditCard          5000 non-null   int64  
dtypes: float64(1), int64(13)
memory usage: 547.0 KB
None

--- İstatistiksel Özet ---


,ID,Age,Experience,Income,ZIP.Code,Family,CCAvg,Education,Mortgage,Personal.Loan,Securities.Account,CD.Account,Online,CreditCard
count,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.000000,5000.00000,5000.000000,5000.000000
mean,2500.500000,45.338400,20.104600,73.774200,93152.503000,2.396400,1.937938,1.881000,56.498800,0.096000,0.104400,0.06040,0.596800,0.294000
std,1443.520003,11.463166,11.467954,46.033729,2121.852197,1.147663,1.747659,0.839869,101.713802,0.294621,0.305809,0.23825,0.490589,0.455637
min,1.000000,23.000000,-3.000000,8.000000,9307.000000,1.000000,0.000000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
25%,1250.750000,35.000000,10.000000,39.000000,91911.000000,1.000000,0.700000,1.000000,0.000000,0.000000,0.000000,0.00000,0.000000,0.000000
50%,2500.500000,45.000000,20.000000,64.000000,93437.000000,2.000000,1.500000,2.000000,0.000000,0.000000,0.000000,0.00000,1.000000,0.000000
75%,3750.250000,55.000000,30.000000,98.000000,94608.000000,3.000000,2.500000,3.000000,101.000000,0.000000,0.000000,0.00000,1.000000,1.000000
max,5000.000000,67.000000,43.000000,224.000000,96651.000000,4.000000,10.000000,3.000000,635.000000,1.000000,1.000000,1.00000,1.000000,1.000000



--- Eksik Değerler ---
Series([], dtype: int64)

--- İlk 5 Satır ---


,ID,Age,Experience,Income,ZIP.Code,Family,CCAvg,Education,Mortgage,Personal.Loan,Securities.Account,CD.Account,Online,CreditCard
0,1,25,1,49,91107,4,1.6,1,0,0,1,0,0,0
1,2,45,19,34,90089,3,1.5,1,0,0,1,0,0,0
2,3,39,15,11,94720,1,1.0,1,0,0,0,0,0,0
3,4,35,9,100,94112,1,2.7,2,0,0,0,0,0,0
4,5,35,8,45,91330,4,1.0,2,0,0,0,0,0,1


In [43]:
# ==========================================
# 1.3 VERİ TEMİZLEME VE ÖN İŞLEME
# ==========================================

# Deneyimi 0'ın altında olanları 0 yap
df.loc[df['Experience'] < 0, 'Experience'] = 0
# Gereksiz sütunları uçur
drop_cols = ["ID", "ZIP.Code"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

X = df.drop(columns=["Personal.Loan"])
y = df["Personal.Loan"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [44]:
# ==========================================
# 1.4 ÖZNİTELİK MÜHENDİSLİĞİ
# ==========================================

numeric_cols = [
    "Age",
    "Experience",
    "Income",
    "Family",
    "CCAvg",
    "Education",
    "Mortgage",
    "Securities.Account",
    "CD.Account",
    "Online",
    "CreditCard",
]

numeric_transformer = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        (
            "poly",
            PolynomialFeatures(
                degree=2, interaction_only=False, include_bias=False
            ),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[("num", numeric_transformer, numeric_cols)],
    remainder="passthrough",
)

pipeline_model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("feature_selection", SelectKBest(score_func=f_classif, k=25)),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=250,
                min_samples_split=4,
                min_samples_leaf=3,
                max_features="sqrt",
                random_state=42,
            ),
        ),
    ]
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("\n" + "=" * 40)
print("--- K-Fold Cross Validation Sonuçları ---")
cv_scores = cross_val_score(pipeline_model, X, y, cv=skf, scoring="accuracy")

for i, score in enumerate(cv_scores, 1):
  print(f"Fold {i}: {score:.4f}")

print(f"\nGerçek K-Fold Başarısı (Ortalama): {np.mean(cv_scores):.4f}")
print(f"Skor Sapması (Standart Sapma)    : {np.std(cv_scores):.4f}")
print("=" * 40 + "\n")


--- K-Fold Cross Validation Sonuçları ---
Fold 1: 0.9900
Fold 2: 0.9890
Fold 3: 0.9820
Fold 4: 0.9870
Fold 5: 0.9820

Gerçek K-Fold Başarısı (Ortalama): 0.9860
Skor Sapması (Standart Sapma)    : 0.0034



In [45]:
# ==========================================
# ARA BÖLÜM: İDEAL K DEĞERİNİ BULMA
# ==========================================
for k_val in [10, 15, 20, 25, 30]:
    temp_pipeline = Pipeline([
        ('preprocessor', preprocessor),
        ('feature_selection', SelectKBest(score_func=f_classif, k=k_val)),
        ("classifier", RandomForestClassifier(n_estimators=250, min_samples_split=4, min_samples_leaf=3, max_features='sqrt', class_weight='balanced', random_state=42)),
    ])

    scores = cross_val_score(temp_pipeline, X, y, cv=skf, scoring='accuracy')
    print(f"k = {k_val} için Ortalama K-Fold Başarısı: {np.mean(scores):.4f}")

k = 10 için Ortalama K-Fold Başarısı: 0.9506
k = 15 için Ortalama K-Fold Başarısı: 0.9650
k = 20 için Ortalama K-Fold Başarısı: 0.9792
k = 25 için Ortalama K-Fold Başarısı: 0.9814
k = 30 için Ortalama K-Fold Başarısı: 0.9808


In [52]:
# ==========================================
# 2. NİHAİ MODEL EĞİTİMİ VE KAYIT
# ==========================================
pipeline_model.fit(X_train, y_train)

y_train_pred = pipeline_model.predict(X_train)
y_pred = pipeline_model.predict(X_test)

print(f"Train Doğruluk Oranı: {accuracy_score(y_train, y_train_pred):.4f}")
print(f"Model Doğruluk Oranı (Accuracy): {accuracy_score(y_test, y_pred):.4f}\n")
print("Sınıflandırma Raporu:")
print(classification_report(y_test, y_pred))



Train Doğruluk Oranı: 0.9948
Model Doğruluk Oranı (Accuracy): 0.9910

Sınıflandırma Raporu:
              precision    recall  f1-score   support

           0       1.00      0.99      1.00       904
           1       0.95      0.96      0.95        96

    accuracy                           0.99      1000
   macro avg       0.97      0.98      0.97      1000
weighted avg       0.99      0.99      0.99      1000

[[1.     0.    ]
 [0.0026 0.9974]
 [1.     0.    ]
 [1.     0.    ]
 [1.     0.    ]]


In [53]:
# ==========================================
# 3. KARA KUTUYU AÇMA: Hangi Özellikler Seçildi?
# ==========================================
preprocessor = pipeline_model.named_steps["preprocessor"]
selector = pipeline_model.named_steps["feature_selection"]
classifier = pipeline_model.named_steps["classifier"]

all_feature_names = preprocessor.get_feature_names_out()
selected_mask = selector.get_support()
selected_features = all_feature_names[selected_mask]

# RandomForest için coef_ yerine feature_importances_ kullanılır
importances = classifier.feature_importances_

feature_importance = pd.DataFrame(
    {
        "Özellik (Feature)": selected_features,
        "Önem Düzeyi (Importance)": importances,
    }
)

# --- GÖRSEL TEMİZLİK VE SIRALAMA BÖLÜMÜ ---
feature_importance["Özellik (Feature)"] = feature_importance[
    "Özellik (Feature)"
].str.replace(r"^(num__|cat__|text__|remainder__)", "", regex=True)

# Önem düzeyine göre azalan şekilde sıralayalım (mutlak değer almaya gerek yok, importances hep pozitiftir)
feature_importance = feature_importance.sort_values(
    by="Önem Düzeyi (Importance)", ascending=False
)

feature_importance["Önem Düzeyi (Importance)"] = feature_importance[
    "Önem Düzeyi (Importance)"
].round(4)
# ------------------------------------------

print("\n--- Modelin Seçtiği En İyi Özellikler ve Önem Düzeyleri ---")
print(feature_importance.to_string(index=False))

print(
    "En düşük tahmin edilen olasılık:",
    pipeline_model.predict_proba(X_test)[:, 1].min(),
)


--- Modelin Seçtiği En İyi Özellikler ve Önem Düzeyleri ---
            Özellik (Feature)  Önem Düzeyi (Importance)
             Income Education                    0.1725
                       Income                    0.1168
                Income Family                    0.0976
            Income CD.Account                    0.0886
                    Education                    0.0693
                        CCAvg                    0.0637
         Education CD.Account                    0.0548
              CCAvg Education                    0.0527
                     Income^2                    0.0469
             CCAvg CD.Account                    0.0398
                 Income CCAvg                    0.0360
                 Family CCAvg                    0.0354
            Family CD.Account                    0.0331
              Income Mortgage                    0.0176
                      CCAvg^2                    0.0166
              Family Mortgage              

In [51]:
# ==========================================
# 4. MODELİ KAYDETME
# ==========================================
os.makedirs("../backend/models", exist_ok=True)
joblib.dump(pipeline_model, "../backend/models/bankloan_pipeline.pkl")
joblib.dump(list(X_train.columns), "../backend/models/model_columns_random_forest.pkl")

print(
    "Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne"
    " kaydedildi!"
)

Pipeline modeli ve sütun isimleri başarıyla 'models/' klasörüne kaydedildi!
